# Phase 03 — Literature & Baseline Research
## CodeMentor-LLM
Testing meta-llama/Meta-Llama-3-8B-Instruct base model on 10 coding questions
before any fine-tuning to justify my project.

#  Libraries

In [ ]:
!pip install -q transformers==4.49.0 bitsandbytes==0.45.3 accelerate==1.5.1 peft==0.14.0

# Checking GPU

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("Memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

# Login to HuggingFace

In [ ]:
from huggingface_hub import login
login()

# Load meta-llama/Meta-Llama-3-8B-Instruct in 4-bit

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

# 4-bit quantization config --> QLoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                     # Load in 4 bits
    bnb_4bit_quant_type="nf4",             # Special format called NormalFloat4 --> better accuracy than normal 4-bit
    bnb_4bit_compute_dtype=torch.bfloat16, # During calculation, use FP16 for stability
    bnb_4bit_use_double_quant=True,        # Compresses even the quantization values
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto", # Split model layers smartly --> GPU and CPU
)

print("Model loaded successfully")
print(f"Memory footprint: {model.get_memory_footprint() / 1024**3:.2f} GB")

# Create inference function

In [ ]:
def generate_response(
    prompt,            # User Question
    max_new_tokens=256 # maximum words/tokens model will generate
    ):

    # Apply Mistral chat template --> This converts your input into chat format.
    messages = [
        {"role": "user",
         "content": prompt}
        ]

    # Tokenize --> Make Token for model preferences
    inputs = tokenizer.apply_chat_template(
        messages,                  # Chat template to be tokenize acc to model prefereneces
        return_tensors="pt",       # Means output becomes tensor (numbers for GPU)
        add_generation_prompt=True # Tell model to start
    ).to("cuda") # Moves data to GPU

    # Generate
    with torch.no_grad(): # Disable gradient calculation ,Saves memory ,Used in inference only --> We are NOT training, only generating
        outputs = model.generate(
            inputs,                             # Input tokens (my prompt)
            max_new_tokens=max_new_tokens,      # Limit output length
            temperature=0.7,                    # Controls randomness
            top_p=0.9,                          # consider only top 90% probability words --> improves quality + removes bad tokens
            do_sample=True,                     # Enables randomness , True → creative output
            pad_token_id=tokenizer.eos_token_id # Fixes padding issue --> eos_token_id = end of sentence token
        )

    # Decode only new tokens
    response = tokenizer.decode(
        outputs[0][inputs.shape[-1]:], # “Ignore the input prompt, keep only new generated text”
        skip_special_tokens=True       # Removes weird tokens
    )
    return response

print("Inference function ready")

# Run 10 coding questions against base model

In [ ]:
# 10 coding questions to test base model
questions = [
    "Write a Python function to reverse a string.",
    "Explain what a decorator is in Python with an example.",
    "What is the difference between a list and a tuple in Python?",
    "Write a Python function to check if a number is prime.",
    "How do you handle exceptions in Python? Show an example.",
    "Write a SQL query to find the second highest salary from a table.",
    "Explain the concept of recursion with a Python example.",
    "What is the difference between == and is in Python?",
    "Write a Python function to find duplicates in a list.",
    "Explain what a REST API is in simple terms.",
]

# Run all 10 questions
results = []
for i, question in enumerate(questions):
    print(f"\n{'='*30}")
    print(f"Q{i+1}: {question}")
    print(f"{'='*30}")
    response = generate_response(question)
    print(f"A: {response}")
    results.append({"question": question, "response": response})

print("\nAll 10 questions completed")

## Base Model Analysis — meta-llama/Meta-Llama-3-8B-Instruct

**Strengths:**
- Answers all 10 questions correctly
- Formats code blocks consistently with backticks
- Gives step-by-step explanations (Q4, Q7)
- Provides multiple solution approaches (Q1 — slice + reversed)
- Uses bold headers to structure answers (Q3, Q8)
- Handles Python, SQL, and conceptual questions
- Good real-world analogies (Q10 — restaurant analogy)
- Proper error handling examples (Q5 — try/except/else)

**Weaknesses:**
- Responses cut off mid-sentence in 6/10 questions (Q2, Q3, Q4, Q7, Q8, Q9)
- No consistent response length
- Attention mask warning on every inference
- No domain-specific coding instruction format
- Sometimes over-explains simple concepts
- Q8 has wrong example — `[1,2,3] == [1,2,3]` returns `True` not `False`
- Q10 wrong full form — "Representational State of Mind" instead of "Representational State Transfer"

**Key Issues Justifying Fine-Tuning:**
1. Responses cut off in 60% of questions — major quality issue
2. Factual errors present (Q8, Q10)
3. No consistent teaching structure
4. SFT will teach complete structured responses
5. DPO will align model to prefer accurate higher quality responses

# Save results to JSON

In [ ]:
import json

# Save baseline results
baseline_results = {
    "model": "meta-llama/Meta-Llama-3-8B-Instruct",
    "type": "base_model_no_finetuning",
    "results": results,
    "observations": {
        "strengths": [
            "Answers basic coding questions correctly",
            "Formats code blocks properly",
            "Gives multiple approaches",
            "Explains concepts clearly"
        ],
        "weaknesses": [
            "Responses get cut off mid-sentence",
            "No consistent response structure",
            "No step-by-step teaching format",
            "Inconsistent response length",
            "Attention mask warning"
        ],
        "finetuning_justified": True
    }
}

# Save to file
with open("baseline_results.json", "w") as f:
    json.dump(baseline_results, f, indent=4)

print("Baseline results saved to baseline_results.json")